In [1]:
%%writefile surveillance_desk.py
import asyncio
import json
import os
import random
import socket
from contextlib import asynccontextmanager
from datetime import datetime
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse

# 1. Base Universe & Statistical Calibration
DEFAULT_BASELINES = {
    "AAPL": {"asset_class": "US_Equities", "last_price": 232.80, "adv_30": 48500000, "atr_14": 3.85, "mean_vol_1m": 125000, "std_vol_1m": 28000},
    "MSFT": {"asset_class": "US_Equities", "last_price": 428.50, "adv_30": 21000000, "atr_14": 5.40, "mean_vol_1m": 56000, "std_vol_1m": 14000},
    "NVDA": {"asset_class": "US_Equities", "last_price": 128.20, "adv_30": 62000000, "atr_14": 4.10, "mean_vol_1m": 165000, "std_vol_1m": 39000},
    "TSLA": {"asset_class": "US_Equities", "last_price": 218.40, "adv_30": 74000000, "atr_14": 6.80, "mean_vol_1m": 195000, "std_vol_1m": 48000},
    "AMZN": {"asset_class": "US_Equities", "last_price": 186.10, "adv_30": 34000000, "atr_14": 3.20, "mean_vol_1m": 90000, "std_vol_1m": 22000},
    "RELIANCE.NS": {"asset_class": "Indian_NSE", "last_price": 2980.50, "adv_30": 6800000, "atr_14": 42.50, "mean_vol_1m": 18000, "std_vol_1m": 4500},
    "TCS.NS": {"asset_class": "Indian_NSE", "last_price": 4210.00, "adv_30": 2400000, "atr_14": 58.00, "mean_vol_1m": 6400, "std_vol_1m": 1600},
    "INFY.NS": {"asset_class": "Indian_NSE", "last_price": 1890.20, "adv_30": 8500000, "atr_14": 28.40, "mean_vol_1m": 22500, "std_vol_1m": 5800},
    "HDFCBANK.NS": {"asset_class": "Indian_NSE", "last_price": 1665.40, "adv_30": 14000000, "atr_14": 22.10, "mean_vol_1m": 37000, "std_vol_1m": 9200},
    "EURUSD=X": {"asset_class": "Forex", "last_price": 1.0845, "adv_30": 1500000, "atr_14": 0.0065, "mean_vol_1m": 4000, "std_vol_1m": 950},
    "GBPUSD=X": {"asset_class": "Forex", "last_price": 1.2980, "adv_30": 1200000, "atr_14": 0.0082, "mean_vol_1m": 3200, "std_vol_1m": 800},
    "USDJPY=X": {"asset_class": "Forex", "last_price": 152.40, "adv_30": 1800000, "atr_14": 1.15, "mean_vol_1m": 4800, "std_vol_1m": 1100},
    "USDINR=X": {"asset_class": "Forex", "last_price": 84.05, "adv_30": 900000, "atr_14": 0.18, "mean_vol_1m": 2400, "std_vol_1m": 550},
    "GC=F": {"asset_class": "Commodities", "last_price": 2745.80, "adv_30": 280000, "atr_14": 26.50, "mean_vol_1m": 750, "std_vol_1m": 180},
    "SI=F": {"asset_class": "Commodities", "last_price": 33.75, "adv_30": 110000, "atr_14": 0.72, "mean_vol_1m": 300, "std_vol_1m": 80},
    "CL=F": {"asset_class": "Commodities", "last_price": 71.40, "adv_30": 420000, "atr_14": 1.95, "mean_vol_1m": 1100, "std_vol_1m": 280},
    "NG=F": {"asset_class": "Commodities", "last_price": 2.58, "adv_30": 190000, "atr_14": 0.12, "mean_vol_1m": 500, "std_vol_1m": 120}
}

if os.path.exists("market_baselines.json"):
    try:
        with open("market_baselines.json", "r") as f:
            loaded = json.load(f)
            if loaded:
                DEFAULT_BASELINES.update(loaded)
    except Exception:
        pass

BASELINES = DEFAULT_BASELINES
SYMBOLS = list(BASELINES.keys())
PRICES = {s: float(BASELINES[s]["last_price"]) for s in SYMBOLS}
CONTROLS = {s: {"status": "NORMAL", "clip_cap": 2500, "throttled": False} for s in SYMBOLS}

# 2. Central WebSocket Dispatcher
class SurveillanceEngine:
    def __init__(self):
        self.active_sockets: list[WebSocket] = []

    async def connect(self, ws: WebSocket):
        await ws.accept()
        self.active_sockets.append(ws)

    def disconnect(self, ws: WebSocket):
        if ws in self.active_sockets:
            self.active_sockets.remove(ws)

    async def broadcast(self, data: dict):
        dead = []
        for ws in self.active_sockets:
            try:
                await ws.send_text(json.dumps(data))
            except Exception:
                dead.append(ws)
        for ws in dead:
            self.disconnect(ws)

    def evaluate(self, sym: str, price: float, vol: float, ctr: float, resting_ms: float):
        meta = BASELINES[sym]
        mean_v = meta.get("mean_vol_1m", 10000)
        std_v = meta.get("std_vol_1m", 2500)
        z_score = (vol - mean_v) / std_v

        price_diff = abs(price - meta["last_price"])
        atr = meta.get("atr_14", 2.0)
        local_time_str = datetime.now().strftime("%H:%M:%S")

        if price_diff > (atr * 2.5):
            CONTROLS[sym]["status"] = "KILL_SWITCH"
            return z_score, {
                "type": "ALERT",
                "symbol": sym,
                "severity": "CRITICAL",
                "typology": "Flash Price Dislocation",
                "speech": f"Critical flash crash detected on {sym}. Kill switch engaged.",
                "metric_details": f"Move: ${price_diff:.2f} (> 2.5x ATR)",
                "control_name": "KILL_SWITCH",
                "control_desc": "⛔ KILL-SWITCH ENGAGED: Purged open book and restricted account.",
                "timestamp": local_time_str
            }

        if ctr >= 0.90 and resting_ms < 200.0:
            CONTROLS[sym]["throttled"] = True
            return z_score, {
                "type": "ALERT",
                "symbol": sym,
                "severity": "HIGH",
                "typology": "Spoofing / Layering Signature",
                "speech": f"Spoofing signature on {sym}. Gateway rate limited.",
                "metric_details": f"CTR: {ctr*100:.1f}% | Avg Resting Life: {resting_ms:.0f}ms",
                "control_name": "THROTTLE",
                "control_desc": "⏳ THROTTLE APPLIED: Rate limited to 5 msgs/sec for 10s cooldown.",
                "timestamp": local_time_str
            }

        if z_score >= 4.0:
            CONTROLS[sym]["status"] = "CLIP_CAP"
            CONTROLS[sym]["clip_cap"] = 500
            return z_score, {
                "type": "ALERT",
                "symbol": sym,
                "severity": "CRITICAL",
                "typology": "Abnormal Volume Burst",
                "speech": f"Volume burst on {sym}. Clip cap restricted.",
                "metric_details": f"Vol Z-Score: +{z_score:.2f}σ | 1M Vol: {int(vol):,}",
                "control_name": "CLIP_CAP",
                "control_desc": "📉 CLIP CAP ENFORCED: Max allowable clip restricted to 500 units.",
                "timestamp": local_time_str
            }

        return z_score, None

engine = SurveillanceEngine()

# 3. Microsecond Market Telemetry Worker
async def market_stream_worker():
    while True:
        try:
            sym = random.choice(SYMBOLS)
            meta = BASELINES[sym]

            atr = meta.get("atr_14", 1.0)
            step = random.gauss(0, atr * 0.012)
            PRICES[sym] = round(max(PRICES[sym] + step, 0.01), 4 if "X" in sym else 2)

            cur_vol = int(meta["mean_vol_1m"] * random.uniform(0.75, 1.35))
            ctr = round(random.uniform(0.10, 0.22), 2)
            resting_ms = round(random.uniform(650.0, 950.0), 1)

            z_score, alert = engine.evaluate(sym, PRICES[sym], cur_vol, ctr, resting_ms)
            local_time_str = datetime.now().strftime("%H:%M:%S")

            tick_msg = {
                "type": "TICK",
                "symbol": sym,
                "asset_class": meta["asset_class"],
                "price": PRICES[sym],
                "volume": cur_vol,
                "z_score": round(z_score, 2),
                "control": CONTROLS[sym]["status"],
                "timestamp": local_time_str
            }
            await engine.broadcast(tick_msg)
            if alert:
                await engine.broadcast(alert)

            await asyncio.sleep(0.35)
        except asyncio.CancelledError:
            break
        except Exception:
            await asyncio.sleep(1.0)

@asynccontextmanager
async def lifespan(app: FastAPI):
    task = asyncio.create_task(market_stream_worker())
    yield
    task.cancel()

app = FastAPI(lifespan=lifespan)

@app.websocket("/ws")
async def websocket_endpoint(ws: WebSocket):
    await engine.connect(ws)
    await ws.send_text(json.dumps({
        "type": "INIT",
        "baselines": BASELINES,
        "prices": PRICES,
        "controls": CONTROLS
    }))
    try:
        while True:
            raw = await ws.receive_text()
            cmd = json.loads(raw)
            action = cmd.get("action")
            target = cmd.get("symbol")

            if action == "INJECT":
                t_type = cmd["typology"]
                meta = BASELINES[target]
                local_time_str = datetime.now().strftime("%H:%M:%S")

                if t_type == "VOLUME_SPIKE":
                    vol = int(meta["mean_vol_1m"] + (meta["std_vol_1m"] * 5.6))
                    z = round((vol - meta["mean_vol_1m"]) / meta["std_vol_1m"], 2)
                    CONTROLS[target]["status"] = "CLIP_CAP"
                    CONTROLS[target]["clip_cap"] = 500
                    await engine.broadcast({
                        "type": "TICK", "symbol": target, "asset_class": meta["asset_class"],
                        "price": PRICES[target], "volume": vol, "z_score": z,
                        "control": "CLIP_CAP", "timestamp": local_time_str
                    })
                    await engine.broadcast({
                        "type": "ALERT", "symbol": target, "severity": "CRITICAL",
                        "typology": "Abnormal Volume Burst",
                        "speech": f"Volume spike detected on {target}. Clip cap clamped to 500 units.",
                        "metric_details": f"Vol Z-Score: +{z}σ | 1M Vol: {vol:,}",
                        "control_name": "CLIP_CAP",
                        "control_desc": "📉 CLIP CAP ENFORCED: Maximum allowable single order clamped to 500 units.",
                        "timestamp": local_time_str
                    })

                elif t_type == "SPOOFING":
                    CONTROLS[target]["throttled"] = True
                    await engine.broadcast({
                        "type": "ALERT", "symbol": target, "severity": "HIGH",
                        "typology": "Spoofing / Layering Signature",
                        "speech": f"Spoofing loop detected on {target}. Rate limiter activated.",
                        "metric_details": "Cancel-To-Replace: 96.2% | Avg Resting Life: 68ms",
                        "control_name": "THROTTLE",
                        "control_desc": "⏳ THROTTLE APPLIED: Rate-limited order gateway to 5 msgs/sec for 10s cooldown.",
                        "timestamp": local_time_str
                    })

                elif t_type == "FLASH_CRASH":
                    drop = meta["atr_14"] * 3.5
                    PRICES[target] = round(PRICES[target] - drop, 2)
                    CONTROLS[target]["status"] = "KILL_SWITCH"
                    await engine.broadcast({
                        "type": "TICK", "symbol": target, "asset_class": meta["asset_class"],
                        "price": PRICES[target], "volume": int(meta["mean_vol_1m"] * 2.8),
                        "z_score": 3.8, "control": "KILL_SWITCH", "timestamp": local_time_str
                    })
                    await engine.broadcast({
                        "type": "ALERT", "symbol": target, "severity": "CRITICAL",
                        "typology": "Flash Price Dislocation",
                        "speech": f"Flash crash on {target}. Kill switch engaged. Trading halted.",
                        "metric_details": f"Price plunged ${drop:.2f} (> 3.5x ATR)",
                        "control_name": "KILL_SWITCH",
                        "control_desc": "⛔ KILL-SWITCH ENGAGED: Halted session and purged active book.",
                        "timestamp": local_time_str
                    })

            elif action == "MANUAL_CONTROL":
                ctrl = cmd.get("control_type")
                if ctrl == "RESET_KILL_SWITCH":
                    CONTROLS[target]["status"] = "NORMAL"
                elif ctrl == "SET_CLIP_CAP":
                    val = int(cmd.get("value", 2500))
                    CONTROLS[target]["clip_cap"] = val
                    CONTROLS[target]["status"] = "NORMAL" if val >= 2000 else "CLIP_CAP"
                elif ctrl == "TOGGLE_THROTTLE":
                    CONTROLS[target]["throttled"] = bool(cmd.get("value", False))

                await engine.broadcast({
                    "type": "CONTROL_UPDATE",
                    "symbol": target,
                    "controls": CONTROLS[target]
                })

    except WebSocketDisconnect:
        engine.disconnect(ws)

@app.get("/")
def render_page():
    return HTMLResponse(DASHBOARD_HTML)

DASHBOARD_HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Cross-Asset Surveillance Desk</title>
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <style>
    * { box-sizing: border-box; }
    body { background: #070b12; color: #f1f5f9; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, monospace; margin: 0; padding: 14px; }
    
    #strobe-banner {
      display: none; background: #991b1b; color: #ffffff; padding: 10px 14px; border-radius: 6px;
      font-weight: 700; margin-bottom: 12px; align-items: center; justify-content: space-between;
      animation: alertPulse 0.9s infinite alternate; font-size: 13px;
    }
    @keyframes alertPulse { from { opacity: 0.85; } to { opacity: 1; filter: brightness(1.3); } }

    header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px; }
    .badge { padding: 4px 10px; border-radius: 999px; font-weight: 700; font-size: 11px; background: #064e3b; color: #34d399; }
    
    .grid { display: grid; grid-template-columns: 2.4fr 1.1fr; gap: 14px; }
    .card { background: #0f172a; border: 1px solid #1e293b; border-radius: 8px; padding: 12px; }

    .sandbox-panel {
      background: #131c31; border: 1px solid #23314d; border-radius: 6px; padding: 10px 12px; margin-top: 10px;
      display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 12px;
    }
    .sandbox-item { font-size: 11px; display: flex; flex-direction: column; gap: 6px; }
    
    .btn { background: #1e293b; border: 1px solid #334155; color: #f8fafc; padding: 5px 10px; border-radius: 4px; font-size: 11px; cursor: pointer; font-weight: 600; }
    .btn:hover { background: #334155; }
    .btn-red { border-color: #ef4444; color: #fca5a5; }
    .btn-red:hover { background: #7f1d1d; }
    .btn-green { border-color: #10b981; color: #6ee7b7; }
    .btn-green:hover { background: #064e3b; }

    .tabs { display: flex; gap: 4px; }
    .tab { background: #1e293b; border: 1px solid #334155; color: #94a3b8; font-size: 10px; padding: 3px 8px; border-radius: 4px; cursor: pointer; font-weight: 600; }
    .tab.active { background: #38bdf8; color: #070b12; }

    table { width: 100%; border-collapse: collapse; font-size: 11px; margin-top: 8px; }
    th, td { padding: 7px 8px; text-align: left; border-bottom: 1px solid #1e293b; }
    th { color: #64748b; text-transform: uppercase; font-size: 10px; }
    tbody tr:hover { background: #1e293b; cursor: pointer; }
    .active-row { border-left: 3px solid #38bdf8; background: #17243c; }

    .pill { padding: 3px 8px; border-radius: 4px; font-weight: 700; font-size: 10px; }
    .pill-normal { background: #064e3b; color: #34d399; }
    .pill-kill { background: #7f1d1d; color: #fca5a5; animation: alertPulse 0.7s infinite alternate; }
    .pill-clip { background: #78350f; color: #fde68a; }

    .feed { max-height: 520px; overflow-y: auto; display: flex; flex-direction: column; gap: 8px; }
    .alert-card {
      background: #1f1118; border-left: 4px solid #ef4444; border-radius: 6px; padding: 10px;
      font-size: 11px; display: flex; flex-direction: column; gap: 4px;
    }
    .alert-card.HIGH { border-left-color: #f59e0b; background: #1d1710; }
    .alert-header { display: flex; justify-content: space-between; font-weight: bold; font-size: 12px; }
    .control-badge { background: #0f172a; padding: 4px 6px; border-radius: 4px; border: 1px solid #334155; font-size: 10px; font-weight: 600; color: #e2e8f0; }
  </style>
</head>
<body>

  <div id="strobe-banner">
    <div style="display:flex; align-items:center; gap:8px;">
      <span style="font-size:16px;">🚨</span>
      <span id="strobe-text">SURVEILLANCE EXCEPTION ENGAGED</span>
    </div>
    <button class="btn btn-green" onclick="dismissStrobe()">Acknowledge & Mute</button>
  </div>

  <header>
    <div>
      <h2 style="margin:0; font-size:18px;">Cross-Asset Automated Surveillance Desk</h2>
      <small style="color:#64748b;">Sub-Second Telemetry | Real-Time Voice Synthesis & Dynamic Risk Execution</small>
    </div>
    <div style="display:flex; align-items:center; gap:12px;">
      <span id="clock" style="font-size:12px; color:#38bdf8; font-weight:bold;">--:--:--</span>
      <label style="font-size:11px; color:#94a3b8; display:flex; align-items:center; gap:4px; cursor:pointer;">
        <input type="checkbox" id="voice-toggle" checked> 🔊 Audio Alert Synthesis
      </label>
      <span id="badge" class="badge">FEED: CONNECTING...</span>
    </div>
  </header>

  <div class="grid">
    <div>
      <div class="card" style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
          <div>
            <span style="color:#94a3b8; font-size:11px; text-transform:uppercase;">Focus Ticker:</span>
            <strong id="focus-sym" style="color:#38bdf8; font-size:15px; margin-left:6px;">NVDA</strong>
          </div>
          <div style="display:flex; gap:6px;">
            <button class="btn btn-red" onclick="inject('VOLUME_SPIKE')">⚡ Inject Volume Burst (+5σ)</button>
            <button class="btn btn-red" onclick="inject('SPOOFING')">⚡ Inject Spoofing Loop</button>
            <button class="btn btn-red" onclick="inject('FLASH_CRASH')">⛔ Inject Flash Crash</button>
          </div>
        </div>

        <canvas id="liveChart" height="85"></canvas>

        <div class="sandbox-panel">
          <div class="sandbox-item">
            <strong style="color:#38bdf8;">Kill-Switch Gate</strong>
            <span id="ctrl-kill-status" style="color:#34d399; font-weight:bold;">ARMED / CLEAR</span>
            <button id="btn-kill-toggle" class="btn btn-red" onclick="resetKillSwitch()" style="display:none;">🔓 Disengage Kill-Switch</button>
          </div>
          <div class="sandbox-item">
            <strong style="color:#f59e0b;">Clip-Cap Enforcement</strong>
            <div>Max Order: <b id="lbl-clip-cap">2,500</b> units</div>
            <input type="range" id="slider-clip" min="200" max="5000" step="100" value="2500" oninput="changeClipCap(this.value)">
          </div>
          <div class="sandbox-item">
            <strong style="color:#a855f7;">Order Rate Throttle</strong>
            <span id="lbl-throttle-status" style="color:#94a3b8;">Normal Gateway (Uncapped)</span>
            <button class="btn" onclick="toggleThrottle()">Toggle Rate Limiter</button>
          </div>
        </div>
      </div>

      <div class="card">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
          <strong style="color:#94a3b8; font-size:11px; text-transform:uppercase;">Perimeter Instruments</strong>
          <div class="tabs">
            <button class="tab active" onclick="filterClass('ALL', this)">ALL</button>
            <button class="tab" onclick="filterClass('US_Equities', this)">US</button>
            <button class="tab" onclick="filterClass('Indian_NSE', this)">NSE</button>
            <button class="tab" onclick="filterClass('Forex', this)">FX</button>
            <button class="tab" onclick="filterClass('Commodities', this)">COMM</button>
          </div>
        </div>
        <table>
          <thead>
            <tr>
              <th>Symbol</th>
              <th>Class</th>
              <th>Price</th>
              <th>1M Vol</th>
              <th>Z-Score</th>
              <th>Control State</th>
              <th>Time</th>
            </tr>
          </thead>
          <tbody id="tb"></tbody>
        </table>
      </div>
    </div>

    <div class="card">
      <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
        <h4 style="margin:0; font-size:12px; text-transform:uppercase; color:#94a3b8;">Real-Time Audit & Anomaly Feed</h4>
        <span id="alert-counter" style="background:#1e293b; padding:2px 8px; border-radius:999px; font-size:10px; font-weight:bold; color:#ef4444;">0 ALERTS</span>
      </div>
      <div class="feed" id="feed">
        <div id="feed-idle" style="color:#475569; font-size:11px; text-align:center; padding:30px 10px;">
          🛡️ Surveillance perimeter armed.<br>Click any simulation button above to test automated risk actions.
        </div>
      </div>
    </div>
  </div>

  <script>
    setInterval(() => {
      document.getElementById('clock').textContent = new Date().toLocaleTimeString();
    }, 1000);

    let baselines = {}, prices = {}, controls = {}, activeSym = "NVDA";
    let alertCount = 0;

    const ctx = document.getElementById('liveChart').getContext('2d');
    const chart = new Chart(ctx, {
      type: 'line',
      data: {
        labels: [],
        datasets: [{
          label: 'NVDA Live Ticks',
          borderColor: '#38bdf8',
          backgroundColor: 'rgba(56, 189, 248, 0.08)',
          fill: true,
          tension: 0.15,
          data: []
        }]
      },
      options: {
        responsive: true,
        scales: {
          x: { ticks: { color: '#64748b', maxRotation: 0 } },
          y: { ticks: { color: '#64748b' } }
        },
        animation: { duration: 100 }
      }
    });

    function speak(text) {
      if (!document.getElementById('voice-toggle').checked) return;
      if ('speechSynthesis' in window) {
        window.speechSynthesis.cancel();
        const utterance = new SpeechSynthesisUtterance(text);
        utterance.rate = 1.05;
        window.speechSynthesis.speak(utterance);
      }
    }

    function setFocus(s) {
      activeSym = s;
      document.getElementById('focus-sym').textContent = s;
      document.querySelectorAll('#tb tr').forEach(r => r.classList.remove('active-row'));
      const activeEl = document.getElementById('r-' + s);
      if(activeEl) activeEl.classList.add('active-row');
      chart.data.datasets[0].label = s + ' Live Ticks';
      chart.data.labels = [];
      chart.data.datasets[0].data = [];
      chart.update();
      updateControlConsole(s);
    }

    function filterClass(cat, btn) {
      document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
      btn.classList.add('active');
      document.querySelectorAll('#tb tr').forEach(r => {
        if(cat === 'ALL' || r.dataset.cat === cat) {
          r.style.display = '';
        } else {
          r.style.display = 'none';
        }
      });
    }

    function updateControlConsole(s) {
      const c = controls[s] || { status: "NORMAL", clip_cap: 2500, throttled: false };
      const killLbl = document.getElementById('ctrl-kill-status');
      const killBtn = document.getElementById('btn-kill-toggle');
      if (c.status === "KILL_SWITCH") {
        killLbl.textContent = "⛔ ENGAGED (HALTED)";
        killLbl.style.color = "#ef4444";
        killBtn.style.display = "block";
      } else {
        killLbl.textContent = "ARMED / CLEAR";
        killLbl.style.color = "#34d399";
        killBtn.style.display = "none";
      }

      document.getElementById('lbl-clip-cap').textContent = Number(c.clip_cap).toLocaleString();
      document.getElementById('slider-clip').value = c.clip_cap;

      const thLbl = document.getElementById('lbl-throttle-status');
      if (c.throttled) {
        thLbl.textContent = "⏳ THROTTLED (5 msgs/sec)";
        thLbl.style.color = "#f59e0b";
      } else {
        thLbl.textContent = "Normal Gateway (Uncapped)";
        thLbl.style.color = "#94a3b8";
      }
    }

    const ws = new WebSocket(`ws://${window.location.host}/ws`);

    ws.onopen = () => {
      const b = document.getElementById('badge');
      b.textContent = 'FEED: ONLINE (350ms)';
      b.style.background = '#064e3b';
      b.style.color = '#34d399';
    };

    ws.onmessage = (e) => {
      const d = JSON.parse(e.data);
      if(d.type === 'INIT') {
        baselines = d.baselines;
        prices = d.prices;
        controls = d.controls;
        const tb = document.getElementById('tb');
        tb.innerHTML = '';
        for(let s in baselines) {
          tb.innerHTML += `
            <tr id="r-${s}" data-cat="${baselines[s].asset_class}" onclick="setFocus('${s}')">
              <td><strong>${s}</strong></td>
              <td style="color:#64748b;">${baselines[s].asset_class}</td>
              <td class="p">${prices[s]}</td>
              <td class="v">-</td>
              <td class="z">-</td>
              <td class="c"><span class="pill pill-normal">NORMAL</span></td>
              <td class="t">-</td>
            </tr>`;
        }
        setFocus(activeSym);
      } 
      else if(d.type === 'TICK') {
        const row = document.getElementById('r-' + d.symbol);
        if(row) {
          row.querySelector('.p').textContent = d.price;
          row.querySelector('.v').textContent = Number(d.volume).toLocaleString();
          const zEl = row.querySelector('.z');
          zEl.textContent = (d.z_score >= 0 ? '+' : '') + d.z_score + 'σ';
          zEl.style.color = d.z_score >= 3.0 ? '#ef4444' : (d.z_score >= 2.0 ? '#f59e0b' : 'inherit');
          
          const ctrlTd = row.querySelector('.c');
          if (d.control === 'KILL_SWITCH') {
            ctrlTd.innerHTML = '<span class="pill pill-kill">⛔ KILL-SWITCH</span>';
          } else if (d.control === 'CLIP_CAP') {
            ctrlTd.innerHTML = '<span class="pill pill-clip">📉 CLIP-CAPPED</span>';
          } else {
            ctrlTd.innerHTML = '<span class="pill pill-normal">NORMAL</span>';
          }
          row.querySelector('.t').textContent = d.timestamp;
        }

        if(d.symbol === activeSym) {
          if(chart.data.labels.length > 25) {
            chart.data.labels.shift();
            chart.data.datasets[0].data.shift();
          }
          chart.data.labels.push(d.timestamp);
          chart.data.datasets[0].data.push(d.price);
          chart.update();
        }
      }
      else if(d.type === 'ALERT') {
        alertCount++;
        document.getElementById('alert-counter').textContent = `${alertCount} ALERTS`;
        
        speak(d.speech);

        const banner = document.getElementById('strobe-banner');
        banner.style.display = 'flex';
        document.getElementById('strobe-text').textContent = `🚨 ${d.severity}: ${d.typology.toUpperCase()} on ${d.symbol} — ${d.metric_details}`;

        const idle = document.getElementById('feed-idle');
        if(idle) idle.remove();

        const feed = document.getElementById('feed');
        const box = document.createElement('div');
        box.className = `alert-card ${d.severity}`;
        box.innerHTML = `
          <div class="alert-header">
            <span style="color:#ef4444;">🚨 ${d.typology}</span>
            <span style="color:#64748b;">${d.timestamp}</span>
          </div>
          <div>Asset: <strong>${d.symbol}</strong> | Trigger: <code style="color:#38bdf8;">${d.metric_details}</code></div>
          <div class="control-badge">${d.control_desc}</div>
        `;
        feed.prepend(box);

        if (d.symbol === activeSym) {
          controls[d.symbol] = controls[d.symbol] || {};
          controls[d.symbol].status = d.control_name;
          updateControlConsole(d.symbol);
        }
      }
      else if(d.type === 'CONTROL_UPDATE') {
        controls[d.symbol] = d.controls;
        if(d.symbol === activeSym) {
          updateControlConsole(d.symbol);
        }
      }
    };

    function inject(typology) {
      if(ws.readyState === WebSocket.OPEN) {
        ws.send(JSON.stringify({ action: "INJECT", symbol: activeSym, typology: typology }));
      }
    }

    function dismissStrobe() {
      document.getElementById('strobe-banner').style.display = 'none';
      if ('speechSynthesis' in window) window.speechSynthesis.cancel();
    }

    function resetKillSwitch() {
      if(ws.readyState === WebSocket.OPEN) {
        ws.send(JSON.stringify({ action: "MANUAL_CONTROL", symbol: activeSym, control_type: "RESET_KILL_SWITCH" }));
      }
    }

    function changeClipCap(val) {
      document.getElementById('lbl-clip-cap').textContent = Number(val).toLocaleString();
      if(ws.readyState === WebSocket.OPEN) {
        ws.send(JSON.stringify({ action: "MANUAL_CONTROL", symbol: activeSym, control_type: "SET_CLIP_CAP", value: val }));
      }
    }

    function toggleThrottle() {
      const cur = controls[activeSym]?.throttled || false;
      if(ws.readyState === WebSocket.OPEN) {
        ws.send(JSON.stringify({ action: "MANUAL_CONTROL", symbol: activeSym, control_type: "TOGGLE_THROTTLE", value: !cur }));
      }
    }
  </script>
</body>
</html>
"""

Writing surveillance_desk.py


In [ ]:
import socket
import uvicorn

# 1. Allow nested event loops inside Jupyter
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    !pip install nest_asyncio -q
    import nest_asyncio
    nest_asyncio.apply()

# 2. Automatically find an open port
def find_available_port(start=8050):
    for port in range(start, start + 50):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', port)) != 0:
                return port
    return start

free_port = find_available_port(8050)

print("=" * 65)
print(f"SURVEILLANCE DESK ACTIVE: http://localhost:{free_port}")
print("=" * 65)

# 3. Run cleanly inside Jupyter's loop
config = uvicorn.Config("surveillance_desk:app", host="0.0.0.0", port=free_port, log_level="info")
server = uvicorn.Server(config)
await server.serve()

SURVEILLANCE DESK ACTIVE: http://localhost:8050


INFO:     Started server process [32564]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8050 (Press CTRL+C to quit)


INFO:     127.0.0.1:62426 - "GET / HTTP/1.1" 200 OK


INFO:     127.0.0.1:62427 - "WebSocket /ws" [accepted]
INFO:     connection open


INFO:     127.0.0.1:62426 - "GET /favicon.ico HTTP/1.1" 404 Not Found
